# Fertility Rate World Map Animation

This notebook documents a simple data visualization project for **World Data Atlas**.

The goal was to create an animated world map showing how fertility rates changed across countries over time.

The visualization uses **World Bank fertility rate data** and maps the values onto country polygons using Python.


## 1. Project idea

The idea was simple:

- collect fertility rate data by country and year
- match each country to a world map using ISO country codes
- color each country based on its fertility rate
- generate one map image per year
- combine the yearly images into an MP4 animation

This first version is intentionally simple.  
There are no complex calculations yet — the main goal was to build the pipeline from data to visual output.


## 2. Data source

The fertility data comes from the **World Bank API**.

Indicator used:

```text
SP.DYN.TFRT.IN
```

Meaning:

```text
Fertility rate, total (births per woman)
```

The data was stored in a SQL Server database in a general World Bank fact table:

```text
worldbank.data
```

Each row represents:

```text
country × indicator × year × value
```

Example structure:

| column | meaning |
|---|---|
| country_code | ISO3 country code |
| country_id | World Bank country ID |
| country_name | country name |
| indicator_code | World Bank indicator code |
| year | year |
| value | fertility rate |


## 3. SQL query

For each year, the script selected fertility values from the database.

Example query:

```sql
SELECT 
    D.[country_code],
    D.[country_id],
    D.[value],
    D.[country_name]
FROM [World_Data_Atlas].[worldbank].[data] AS D
LEFT JOIN (
    SELECT 
        [wb_id],
        [iso2_code],
        [name],
        [is_country]
    FROM [World_Data_Atlas].[worldbank].[entities]
    WHERE [is_country] = 1
) AS ENT 
    ON ENT.[name] = D.[country_name]
WHERE 
    D.[indicator_code] = 'SP.DYN.TFRT.IN'
    AND D.[year] = 2024
    AND ENT.[is_country] = 1
```

The output was then converted into a Python dictionary:

```python
highlight_countries = {
    "FRA": 1.79,
    "USA": 1.66,
    ...
}
```


## 4. Map data

The country borders were taken from **Natural Earth**:

```text
https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip
```

The map was loaded with GeoPandas:

```python
world = gpd.read_file(WORLD_URL)
```

To make the world map look more natural, the geometries were transformed into the Robinson projection:

```python
world = world.to_crs("+proj=robin")
```

Countries were matched using ISO3 codes, mainly through the Natural Earth column:

```text
ADM0_A3
```


## 5. Visualization method

The map was created using:

- `geopandas`
- `pandas`
- `matplotlib`

The basic logic:

1. Load world country polygons.
2. Create a dataframe with fertility values.
3. Merge the fertility dataframe with the map polygons.
4. Fill missing countries with a default value.
5. Plot countries using a color scale.
6. Save one PNG image per year.

For the animation, a fixed color scale was used:

```python
legend_min_value = 0
legend_max_value = 9
```

This is important because otherwise the colors would change meaning from year to year.


In [ ]:
# Example: convert SQL result into map input

# df = pd.read_sql(query, engine)

# highlight_countries = (
#     df.dropna(subset=["country_code", "value"])
#       .set_index("country_code")["value"]
#       .to_dict()
# )

# highlight_countries example:
highlight_countries = {
    "FRA": 1.8,
    "USA": 1.7,
    "NGA": 5.2,
    "IND": 2.0,
}


## 6. Creating yearly frames

The visualization was generated in a loop from 1960 to 2024.

Simplified example:

```python
for year in range(1960, 2025):
    # get fertility data for selected year
    # convert result to highlight_countries dictionary
    # create map
    # save image as PNG
```

Each frame was saved into a folder:

```text
world_fertility/
```

Example output:

```text
world_fertility_1960.png
world_fertility_1961.png
...
world_fertility_2024.png
```


## 7. Creating the video

The saved PNG images were combined into an MP4 video.

The final video had to be encoded in a format suitable for X/Twitter:

- MP4
- H.264 codec
- yuv420p pixel format
- even image dimensions
- reasonable resolution

This avoids upload problems on social platforms.


In [ ]:
from pathlib import Path
from moviepy import ImageSequenceClip

BASE_DIR = Path.cwd()
input_folder = BASE_DIR / "world_fertility"
output_file = BASE_DIR / "fertility_x.mp4"

images = sorted(str(p) for p in input_folder.glob("*.png"))

print("Frames found:", len(images))

# clip = ImageSequenceClip(images, fps=10)

# clip.write_videofile(
#     str(output_file),
#     codec="libx264",
#     audio=False,
#     preset="medium",
#     bitrate="5000k",
#     ffmpeg_params=[
#         "-vf", "scale=1920:-2",
#         "-pix_fmt", "yuv420p",
#         "-movflags", "+faststart"
#     ]
# )


## 8. Result

The final output is an animated choropleth map showing global fertility rates from **1960 to 2024**.

Even in this simple first version, the main global pattern is visible:

> fertility rates have declined across much of the world over time.

This project is a first step toward a broader data visualization pipeline for World Data Atlas.


## 9. Limitations

This version is intentionally basic.

Current limitations:

- no deep statistical analysis yet
- no uncertainty handling
- limited explanation of regional differences
- simple color scale
- only one World Bank indicator
- no interactive dashboard yet

Future versions can add:

- GDP comparison
- regional aggregation
- fertility decline ranking
- animated labels
- interactive maps
- more refined design


## 10. Tools used

Main tools:

```text
Python
Pandas
GeoPandas
Matplotlib
MoviePy
SQL Server
World Bank API
Natural Earth map data
```

This notebook serves as a lightweight methodology note for the first fertility animation published under the World Data Atlas project.
